# Введение в MapReduce модель на Python


In [2]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [3]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [15]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 1.893626512826196),
 (1, 1.893626512826196),
 (2, 1.893626512826196),
 (3, 1.893626512826196),
 (4, 1.893626512826196)]

## Inverted index 

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [20]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [21]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('it', 18), ('what', 10)]),
 (1, [('a', 2), ('is', 18)])]

## TeraSort

In [22]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, 0.0012754402788138774),
   (None, 0.03674817889665227),
   (None, 0.05198114842720303),
   (None, 0.10057384644390743),
   (None, 0.19260810471044254),
   (None, 0.22071933363088891),
   (None, 0.2628629575008188),
   (None, 0.2887743220077359),
   (None, 0.35400849672742085),
   (None, 0.38921465218454965),
   (None, 0.40745033793381524),
   (None, 0.4884279649516976)]),
 (1,
  [(None, 0.5159810826765979),
   (None, 0.519607661037252),
   (None, 0.556818953000479),
   (None, 0.5969324275082397),
   (None, 0.6047870280429845),
   (None, 0.6191973962237997),
   (None, 0.6249643812712575),
   (None, 0.6604261165888173),
   (None, 0.7006044686711087),
   (None, 0.7169874038961497),
   (None, 0.7228151985941792),
   (None, 0.7921724326159645),
   (None, 0.8104063670891903),
   (None, 0.8297997415803705),
   (None, 0.9071986570742654),
   (None, 0.9236116031860753),
   (None, 0.9687740456504207),
   (None, 0.9884074310184141)])]

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [23]:
data = [7, 2, 19, 4, 11, 3]

def RECORDREADER():
    for x in data:
        yield ("max_key", x)

def MAP(k, v):
    yield (k, v)

def REDUCE(k, values):
    current_max = float('-inf')
    for val in values:
        if val > current_max:
            current_max = val
    yield (k, current_max)

result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(result)

[('max_key', 19)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [24]:
data = [7, 2, 19, 4, 11, 3]

def RECORDREADER():
    for x in data:
        yield ("avg_key", x)

def MAP(k, v):
    yield (k, (v, 1))

def REDUCE(k, values):
    total_sum = 0
    total_count = 0
    for val, cnt in values:
        total_sum += val
        total_count += cnt
    yield (k, total_sum / total_count if total_count else 0)

result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(result)

[('avg_key', 7.666666666666667)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [25]:
def groupbykey_sorted(pairs):
    pairs_sorted = sorted(pairs, key=lambda x: x[0])
    
    result = []
    current_key = None
    current_group = []
    
    for key, value in pairs_sorted:
        if key != current_key:
            if current_key is not None:
                result.append((current_key, current_group))
            current_key = key
            current_group = [value]
        else:
            current_group.append(value)
    
    if current_key is not None:
        result.append((current_key, current_group))
    
    return result


# тест
test_data = [(1, 'a'), (2, 'b'), (1, 'c'), (2, 'd'), (3, 'e')]
print(groupbykey_sorted(test_data))

[(1, ['a', 'c']), (2, ['b', 'd']), (3, ['e'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [26]:
data = [1, 2, 2, 3, 4, 4, 5, 5, 5]

maps = 2
reducers = 2

def INPUTFORMAT():
    chunk_size = len(data) // maps + 1
    for i in range(0, len(data), chunk_size):
        yield [(x, None) for x in data[i:i+chunk_size]]

def MAP(k, _):
    yield (k, 1)

def REDUCE(k, values):
    yield (k, None)

result = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)
result = [(pid, list(part)) for pid, part in result]
print(result)

9 key-value pairs were sent over a network.
[(0, [(2, None), (4, None)]), (1, [(1, None), (3, None), (5, None)])]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [27]:
data = [
    {"id": 1, "age": 25},
    {"id": 2, "age": 35},
    {"id": 3, "age": 40}
]

def RECORDREADER():
    for t in data:
        yield (None, t)

def MAP(_, t):
    if t["age"] > 30:
        yield (tuple(t.items()), tuple(t.items()))

def REDUCE(_, values):
    for v in values:
        yield v

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(('id', 2), ('age', 35)), (('id', 3), ('age', 40))]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [28]:
data = [
    {"id": 1, "age": 25, "name": "A"},
    {"id": 1, "age": 25, "name": "B"}
]

def RECORDREADER():
    for t in data:
        yield (None, t)

def MAP(_, t):
    t_proj = (t["id"], t["age"])
    yield (t_proj, t_proj)

def REDUCE(key, values):
    yield (key, key)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[((1, 25), (1, 25))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [29]:
R = [1, 2, 3]
S = [3, 4, 5]

def RECORDREADER():
    for t in R:
        yield (None, t)
    for t in S:
        yield (None, t)

def MAP(_, t):
    yield (t, t)

def REDUCE(key, values):
    yield (key, key)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(1, 1), (2, 2), (3, 3), (4, 4), (5, 5)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [30]:
R = [1, 2, 3]
S = [2, 3, 4]

def RECORDREADER():
    for t in R:
        yield (None, t)
    for t in S:
        yield (None, t)

def MAP(_, t):
    yield (t, t)

def REDUCE(t, values):
    vals = list(values)
    if len(vals) > 1:
        yield (t, t)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(2, 2), (3, 3)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [31]:
R = [1, 2, 3]
S = [2, 3, 4]

def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(source, t):
    yield (t, source)

def REDUCE(t, sources):
    src = list(sources)
    if src == ["R"]:
        yield (t, t)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(1, 1)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [32]:
R = [(1, "x"), (2, "y"), (3, "z")]
S = [("x", 10), ("y", 20), ("w", 30)]

def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(source, t):
    if source == "R":
        a, b = t
        yield (b, ("R", a))
    else:
        b, c = t
        yield (b, ("S", c))

def REDUCE(b, values):
    left = []
    right = []
    
    for tag, val in values:
        if tag == "R":
            left.append(val)
        else:
            right.append(val)
    
    for a in left:
        for c in right:
            yield (a, b, c)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[(1, 'x', 10), (2, 'y', 20)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [33]:
data = [
    ("A", 10, 1),
    ("A", 20, 2),
    ("B", 5, 3),
    ("B", 15, 4)
]

def RECORDREADER():
    for t in data:
        yield (None, t)

def MAP(_, t):
    a, b, c = t
    yield (a, b)

def REDUCE(a, values):
    total = 0
    for v in values:
        total += v
    yield (a, total)

print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

[('A', 30), ('B', 20)]


# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [34]:
import numpy as np

matrix = np.random.rand(3, 4)
vector = np.random.rand(4)

def RECORDREADER():
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            yield ((i, j), (matrix[i, j], vector[j]))

def MAP(key, value):
    i, j = key
    m_ij, v_j = value
    yield (i, m_ij * v_j)

def REDUCE(i, values):
    yield (i, sum(values))

# проверка
result = dict(MapReduce(RECORDREADER, MAP, REDUCE))
reference = matrix.dot(vector)

print("MapReduce:", result)
print("Numpy:", reference)

MapReduce: {0: 1.2250767919803924, 1: 0.5286454874619522, 2: 0.8257422164192689}
Numpy: [1.22507679 0.52864549 0.82574222]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [35]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [36]:
import numpy as np

I = 2
J = 3
K = 4*10

small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1
    
    # solution code that yield(k2,v2) pairs
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i][j] * w)

def REDUCE(key, values):
    (i, k) = key
    
    # solution code that yield(k3,v3) pairs
    total = 0
    for v in values:
        total += v
    yield (key, total)

Проверьте своё решение

In [37]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [38]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Max i:", max(i for ((i,k), v) in reduce_output))

Max i: 1


Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [39]:

import numpy as np

I, J, K = 3, 4, 50

M = np.random.rand(I, J)
N = np.random.rand(J, K)

def RECORDREADER_2():
    for i in range(I):
        for j in range(J):
            yield ("M", (i, j, M[i, j]))
    for j in range(J):
        for k in range(K):
            yield ("N", (j, k, N[j, k]))

def MAP_2(source, value):
    if source == "M":
        i, j, v = value
        yield (j, ("M", i, v))
    else:
        j, k, w = value
        yield (j, ("N", k, w))

def REDUCE_2(j, values):
    M_vals, N_vals = [], []

    for tag, idx, val in values:
        if tag == "M":
            M_vals.append((idx, val))
        else:
            N_vals.append((idx, val))

    for i, v in M_vals:
        for k, w in N_vals:
            yield ((i, k), v * w)


# стадия 2 (суммирование)
def MAP_2_2(key, value):
    yield (key, value)

def REDUCE_2_2(key, values):
    yield (key, sum(values))


# запуск
intermediate = MapReduce(RECORDREADER_2, MAP_2, REDUCE_2)
result = list(MapReduce(lambda: intermediate, MAP_2_2, REDUCE_2_2))

print("Result size:", len(result))
print("First 10:", result[:10])

# проверка
ref = np.matmul(M, N)
print("Correct:", np.allclose(ref, asmatrix(result)))

Result size: 150
First 10: [((0, 0), 0.4204718780951779), ((0, 1), 0.269260175731675), ((0, 2), 0.37821535402157747), ((0, 3), 0.7356079972446097), ((0, 4), 0.8920951325359809), ((0, 5), 0.6460470212010615), ((0, 6), 0.7463761374749189), ((0, 7), 0.549504403151256), ((0, 8), 0.32334634177486354), ((0, 9), 0.7015849073782202)]
Correct: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [40]:
import numpy as np

I, J, K = 3, 4, 50

small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    if not reduce_output:
        return np.array([])
    I_max = max(i for ((i, k), vw) in reduce_output) + 1
    K_max = max(k for ((i, k), vw) in reduce_output) + 1
    mat = np.zeros((I_max, K_max))
    for ((i, k), vw) in reduce_output:
        mat[i, k] = vw
    return mat

def INPUTFORMAT_3():
    data = []
    
    # small matrix
    for i in range(I):
        for j in range(J):
            data.append((('S', i, j), small_mat[i, j]))
    
    # big matrix
    for j in range(J):
        for k in range(K):
            data.append((('B', j, k), big_mat[j, k]))
    
    yield data

def MAP_3(k, v):
    tag, a, b = k
    if tag == 'S':
        # (i, j) -> ключ j, значение ('S', i, v)
        yield (b, ('S', a, v))
    else:
        # (j, k) -> ключ j, значение ('B', k, v)
        # a это j, b это k
        yield (a, ('B', b, v))

def REDUCE_3(j, values):
    S_vals = []
    B_vals = []
    
    for tag, idx, val in values:
        if tag == 'S':
            S_vals.append((idx, val))
        else:
            B_vals.append((idx, val))
    
    for i, v in S_vals:
        for k, w in B_vals:
            yield ((i, k), v * w)

# стадия суммирования
def MAP_3_2(k, v):
    yield (k, v)

def REDUCE_3_2(k, values):
    yield (k, sum(values))

# запуск
data = []
for part in INPUTFORMAT_3():
    data.extend(part)

intermediate = MapReduce(lambda: data, MAP_3, REDUCE_3)
result = list(MapReduce(lambda: intermediate, MAP_3_2, REDUCE_3_2))

print("Distributed result (first 10):", result[:10])

ref = np.matmul(small_mat, big_mat)
print("Correct:", np.allclose(ref, asmatrix(result)))

Distributed result (first 10): [((0, 0), 1.046174677181986), ((0, 1), 0.7682282404727112), ((0, 2), 1.6152095335579482), ((0, 3), 1.6995128897837355), ((0, 4), 1.888886287941669), ((0, 5), 0.7572315509608754), ((0, 6), 1.7268039671060784), ((0, 7), 1.870341053920424), ((0, 8), 1.2206610372606095), ((0, 9), 1.090103025006122)]
Correct: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [41]:
def INPUTFORMAT_4():
    chunk = 2
    parts = []
    
    # small matrix - разбиваем по строкам
    for start_i in range(0, I, chunk):
        part = []
        for i in range(start_i, min(start_i + chunk, I)):
            for j in range(J):
                part.append((('S', i, j), small_mat[i, j]))
        parts.append(part)
    
    # big matrix - разбиваем по строкам (индекс j)
    for start_j in range(0, J, chunk):
        part = []
        for j in range(start_j, min(start_j + chunk, J)):
            for k in range(K):
                part.append((('B', j, k), big_mat[j, k]))
        parts.append(part)
    
    for p in parts:
        yield p

# собираем ВСЕ данные
data = []
for part in INPUTFORMAT_4():
    data.extend(part)

# тот же pipeline
intermediate = MapReduce(lambda: data, MAP_3, REDUCE_3)
result = list(MapReduce(lambda: intermediate, MAP_3_2, REDUCE_3_2))

print("Multi-reader result (first 10):", result[:10])

ref = np.matmul(small_mat, big_mat)
print("Correct:", np.allclose(ref, asmatrix(result)))

Multi-reader result (first 10): [((0, 0), 1.046174677181986), ((0, 1), 0.7682282404727112), ((0, 2), 1.6152095335579482), ((0, 3), 1.6995128897837355), ((0, 4), 1.888886287941669), ((0, 5), 0.7572315509608754), ((0, 6), 1.7268039671060784), ((0, 7), 1.870341053920424), ((0, 8), 1.2206610372606095), ((0, 9), 1.090103025006122)]
Correct: True


# Ответ на вопрос
Решение не будет работать корректно, если RECORDREADER-ы генерируют случайное подмножество элементов.
Причина:
для вычисления
$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$
необходимо наличие всех значений по индексу j.
Если часть элементов отсутствует — сумма будет неполной, и результат неверный.